# Fine-Tuning Qwen3-TTS

Runs SFT training on the prepared JSONL, tests the checkpoint,
and synthesises the full story script with the fine-tuned voice.

**Prerequisite:** run `data_prep.ipynb` first — `finetune/train_with_codes.jsonl` must exist.

## 1. Setup

In [1]:
import os, json, warnings, subprocess, sys
import torch, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from pydub import AudioSegment
from safetensors.torch import load_file, save_file
from qwen_tts import Qwen3TTSModel

warnings.filterwarnings("ignore")

# ── config ────────────────────────────────────────────────────────────────
REF_AUDIO    = Path("audio/ref_en.wav")
OUTPUT_DIR   = Path("output")
CKPT_DIR     = Path("finetune/checkpoints")
OUT_CODES    = Path("finetune/train_with_codes.jsonl")
SCRIPT_PATH  = Path("script/llm_story.txt")
SPEAKER_NAME = "laxmikant"
# ──────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


## 2. Prepare & Train

In [2]:
ref_24k = REF_AUDIO.parent / (REF_AUDIO.stem + "_24k.wav")
if not ref_24k.exists():
    seg = AudioSegment.from_wav(str(REF_AUDIO))
    seg = seg.set_frame_rate(24000)
    seg = seg.set_channels(1)
    seg.export(str(ref_24k), format="wav")

lines   = OUT_CODES.read_text(encoding="utf-8").splitlines()
patched = []
for line in lines:
    entry = json.loads(line)
    entry["ref_audio"] = str(ref_24k.resolve())
    patched.append(json.dumps(entry, ensure_ascii=False))

OUT_CODES.write_text("\n".join(patched) + "\n", encoding="utf-8")

571311

In [3]:
env = os.environ.copy()
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

result = subprocess.run(
    [
        sys.executable, "finetune/scripts/finetuning/sft_12hz.py",
        "--init_model_path",   "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
        "--output_model_path", str(CKPT_DIR),
        "--train_jsonl",       str(OUT_CODES),
        "--batch_size",        "4",
        "--lr",                "2e-6",
        "--num_epochs",        "5",
        "--speaker_name",      SPEAKER_NAME,
    ],
    capture_output=True, text=True, env=env,
)

print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print(result.stderr[-2000:])


********
********
 
Epoch 0 | Step 0 | Loss: 14.1498
Epoch 0 | Step 10 | Loss: 13.2580
Epoch 0 | Step 20 | Loss: 12.7612
Epoch 1 | Step 0 | Loss: 9.2194
Epoch 1 | Step 10 | Loss: 12.1618
Epoch 1 | Step 20 | Loss: 9.7829
Epoch 2 | Step 0 | Loss: 9.1876
Epoch 2 | Step 10 | Loss: 10.6663
Epoch 2 | Step 20 | Loss: 10.3947
Epoch 3 | Step 0 | Loss: 10.0048
Epoch 3 | Step 10 | Loss: 10.3409
Epoch 3 | Step 20 | Loss: 7.6859
Epoch 4 | Step 0 | Loss: 10.1920
Epoch 4 | Step 10 | Loss: 9.6157
Epoch 4 | Step 20 | Loss: 9.4569



## 3. Test the Fine-Tuned Model

In [4]:
CHECKPOINT = "checkpoint-epoch-4"   # ← change to whichever checkpoint to test

ft_model = Qwen3TTSModel.from_pretrained(str((CKPT_DIR / CHECKPOINT).resolve()), device_map="cuda:0", dtype=torch.bfloat16)

wavs, sr = ft_model.generate_custom_voice(
    text="The voice cloning system is working correctly. This is a quick sanity check.",
    language="English",
    speaker=SPEAKER_NAME,
    instruct="Speak naturally and clearly at a moderate pace.",
)

sf.write(str(OUTPUT_DIR / "finetuned_test.wav"), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


## 4. Synthesize Full Script

In [5]:
# ── knobs ──────────────────────────────────────────────────────────────────
CHECKPOINT  = "checkpoint-epoch-2"   # ← change to whichever checkpoint to use
TEMPERATURE = 0.72                   # lower = more consistent voice
# ──────────────────────────────────────────────────────────────────────────

synth_model = Qwen3TTSModel.from_pretrained(str((CKPT_DIR / CHECKPOINT).resolve()), device_map="cuda:0", dtype=torch.bfloat16)

text = SCRIPT_PATH.read_text(encoding="utf-8").strip()

wavs, sr = synth_model.generate_custom_voice(
    text=text,
    language="English",
    speaker=SPEAKER_NAME,
    instruct="Warm, engaging storytelling tone. Natural pace with gentle variation in pitch.",
    temperature=TEMPERATURE,
    top_p=0.88,
    top_k=50,
    repetition_penalty=1.1,
    max_new_tokens=4096,
)

out_path = OUTPUT_DIR / f"{SCRIPT_PATH.stem}_finetuned.wav"
sf.write(str(out_path), wavs[0], sr)
ipd.display(ipd.Audio(wavs[0], rate=sr))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
